# Attention 注意力机制

这里主要介绍 Transformer 中最核心的 scaled dot-product attention，也就是缩放点积注意力。

Attention 想解决的问题可以这样理解：对于序列中的每一个 token，我们希望它不要只看自己，而是能够根据当前任务，从序列中的其他 token 中取回有用的信息。

## Q、K、V 是什么

在 self-attention 中，输入是同一个序列 embedding：

$$
X \in \mathbb{R}^{B \times T \times d_{model}}
$$

其中 $B$ 是 batch size，$T$ 是 sequence length，$d_{model}$ 是每个 token 的向量维度。

Attention 会先通过线性变换得到三个张量：Query、Key 和 Value：

$$
Q = XW_Q
$$

$$
K = XW_K
$$

$$
V = XW_V
$$

其中：

$$
Q \in \mathbb{R}^{B \times T \times d_k}, \quad
K \in \mathbb{R}^{B \times T \times d_k}, \quad
V \in \mathbb{R}^{B \times T \times d_v}
$$

在本 notebook 的简化实现中，使用一个 `nn.Linear(hidden_size, hidden_size * 3)` 一次性生成 Q、K、V，然后再把最后一维拆成三份。因此这里是一个单头 self-attention 的实现，并且代码中有：

$$
d_k = d_v = d_{model} = hidden\_size
$$

## Scaled Dot-Product Attention

缩放点积注意力的标准公式是：

$$
\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

注意这里的缩放因子 $\sqrt{d_k}$ 是在 softmax 之前除到 attention scores 上，而不是在乘以 $V$ 之后再除。

整个计算过程可以分成三步。

第一步，计算相关性分数：

$$
S = \frac{QK^T}{\sqrt{d_k}}
$$

对于 batch 中的每个样本，$QK^T$ 会得到一个 $T \times T$ 的矩阵。因此：

$$
S \in \mathbb{R}^{B \times T \times T}
$$

其中 $S_{i,j}$ 表示第 $i$ 个 token 在读取信息时，对第 $j$ 个 token 的关注程度。

第二步，对最后一个维度做 softmax，把分数变成权重：

$$
A = \operatorname{softmax}(S)
$$

因此：

$$
\sum_{j=1}^{T} A_{i,j} = 1
$$

也就是说，对于每一个 query token，它会得到一组针对所有 key token 的注意力分布。

第三步，用注意力权重对 Value 做加权求和：

$$
O = AV
$$

输出 shape 为：

$$
O \in \mathbb{R}^{B \times T \times d_v}
$$

如果 $d_v = d_{model}$，那么输出会保持和输入相同的最后一维大小。

## 一个直观类比

可以把 attention 理解成在图书馆中查资料：

- $Q$ 是你当前的问题：我想找什么？
- $K$ 是每本书的索引：这本书主要和什么相关？
- $V$ 是书里的实际内容：如果这本书相关，我最终要取回什么信息？

先用 $QK^T$ 判断“问题”和“索引”的匹配程度，再通过 softmax 得到查阅每本书的比例，最后根据这个比例从 $V$ 中加权取回信息。

## 为什么要除以 $\sqrt{d_k}$

当 $d_k$ 比较大时，$QK^T$ 中每个元素都是多个维度点积相加的结果，数值方差会随着维度增大而变大。分数过大时，softmax 会变得非常尖锐，容易进入梯度很小的饱和区域。

用 $\sqrt{d_k}$ 做缩放，可以让 attention scores 的数值范围更稳定：

$$
S = \frac{QK^T}{\sqrt{d_k}}
$$

这也是 Transformer 原论文 Attention Is All You Need 中使用 scaled dot-product attention 的原因。

## Mask 的作用

在一些任务中，并不是所有 token 都应该互相看到。例如在自回归语言模型中，第 $t$ 个 token 只能看到自己和它之前的 token，不能看到未来 token。这个时候就需要 mask。

在代码中，mask 会作用在 softmax 之前的 scores 上：

$$
S_{masked} = \operatorname{mask}(S)
$$

常见做法是把不能看的位置设成一个非常小的数，例如 $-10^9$。这样经过 softmax 后，这些位置的权重会接近 0。

本 notebook 下面的例子使用了一个下三角 mask：

$$
M = \operatorname{tril}(\mathbf{1}_{T \times T})
$$

它表示每个 token 只能关注自己和前面的 token。


In [ ]:
# Attention 实现
import torch
import torch.nn as nn
import numpy as np

class Attention():
    def __init__(self, hidden_size):
        self.hidden_size = hidden_size

        self.W_qkv = nn.Linear(hidden_size, hidden_size * 3)
        self.softmax = nn.Softmax(dim=-1)
        self.scale = np.sqrt(hidden_size)
    
    def forward(self, x, mask=None):
        # x: [batch_size, seq_len, hidden_size]
        # mask: [batch_size, seq_len, seq_len] or None
        batch_size, seq_len, hidden_size = x.size()

        # 计算 Q, K, V
        qkv = self.W_qkv(x)  # [batch_size, seq_len, hidden_size * 3]
        qkv = qkv.view(batch_size, seq_len, 3, hidden_size)  # [batch_size, seq_len, 3, hidden_size]
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]  # [batch_size, seq_len, hidden_size]

        # 计算注意力分数
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.scale  # [batch_size, seq_len, seq_len]
        # 做 masking
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention_weights = self.softmax(scores)  # [batch_size, seq_len, seq_len]

        # 源代码里这里会对 attention_weights 做 dropout, 在 Dropout 更新后补全
        # TODO: Add dropout if needed

        # 加权求和
        output = torch.matmul(attention_weights, v)  # [batch_size, seq_len, hidden_size]

        return output, attention_weights


In [14]:
# a simple sample to test the Attention implementation
x = torch.rand(2, 5, 8)  # [batch_size=2, seq_len=5, hidden_size=8]
# 假设每个 token 只能看见自己和前面的 token，后面的 token 被 mask 掉
mask = torch.tril(torch.ones(5, 5)).unsqueeze(0).repeat(2, 1, 1)
print("Mask:\n", mask)

Mask:
 tensor([[[1., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 0.],
         [1., 1., 1., 1., 1.]],

        [[1., 0., 0., 0., 0.],
         [1., 1., 0., 0., 0.],
         [1., 1., 1., 0., 0.],
         [1., 1., 1., 1., 0.],
         [1., 1., 1., 1., 1.]]])


In [15]:
attention = Attention(hidden_size=8)
output, attention_weights = attention.forward(x, mask)
print("Output:\n", output.shape)  # [batch_size=2, seq_len=5, hidden_size=8]
print("Attention Weights:\n", attention_weights)  # [batch_size=2, seq_len=5, seq_len=5]

Output:
 torch.Size([2, 5, 8])
Attention Weights:
 tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5002, 0.4998, 0.0000, 0.0000, 0.0000],
         [0.3253, 0.3333, 0.3414, 0.0000, 0.0000],
         [0.2416, 0.2456, 0.2559, 0.2569, 0.0000],
         [0.1831, 0.1950, 0.2202, 0.2076, 0.1941]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5075, 0.4925, 0.0000, 0.0000, 0.0000],
         [0.3252, 0.3330, 0.3418, 0.0000, 0.0000],
         [0.2646, 0.2520, 0.2479, 0.2355, 0.0000],
         [0.2117, 0.2033, 0.1977, 0.1921, 0.1951]]],
       grad_fn=<SoftmaxBackward0>)
